# Notebook 17 — what are the 300 scored items actually made of?

`transcription_correct` is an exact match between two canonicalized strings, so
the perception AUROC rests on that comparison being fair. This notebook opens
the comparison up, classifies **every** item by why it scored the way it did,
and shows the handwriting behind each category.

For every item the scoring pipeline runs four stages, on the model samples and
on the ground truth alike:

| stage | function | produces |
|---|---|---|
| 1 | *(the run)* | raw model output |
| 2 | `pilot.parsing.parse_transcription` | the `**Answer:**` field |
| 3 | `pilot.canonicalize.extract_final_answer` | the final-answer span |
| 4 | `pilot.rescore.answer_label` | the string actually compared |

Offline inspection found four things worth seeing rather than taking on
trust, three of them genuine bugs:

1. **A real bug.** `structural_clean` unwraps `\textcolor{}{}` with a `[^{}]*`
   body, so nested cases survive into the label. 85/300 ground truths, 59 of
   them `has_error=1` — FERMAT marks the *injected error* in red.
2. **A second real bug.** `extract_final_answer`'s last-line tier splits on
   `.` as a sentence terminator, so it also splits decimal numbers:
   `"the area is 75.46 cm."` extracts as `"46 cm"`.
3. **A third real bug.** `parse_latex` does not fail loudly on input it only
   partly understands — it parses a **prefix** and silently returns it.
   `"Hence, the required number of words is 24"` becomes `h*(e*(n*(c*e)))`
   (SymPy read "Hence" as five multiplied variables and threw the 24 away);
   `"40^\circ 20' = \frac{121\pi}{540}"` becomes `40**circ*20`, dropping the
   `=` and the entire answer. 37/300 ground truths, 44/1500 samples.
4. **Formatting and scope mismatches.** `2^3 = 8` vs `2^{3} = 8`; `0 = 9` vs
   `0 = 9,`; and the model reporting `= 75.46 cm^2` where the ground truth
   spells out `= \pi r^2 = ... = 75.46 cm^2`.

Bugs 2 and 3 are why they had to be *fixed* rather than noted. Bug 3 is the
worst because it **collapses**: `\frac{1210}{540}` and `\frac{121\pi}{540}`
are different answers that reduce to the same label, which deflates entropy as
well as manufacturing matches. Fixing them *lowers* accuracy — the honest
direction.

**No GPU, no model.** Reads two results CSVs and the FERMAT images. ~5 min.

**What this notebook does NOT do:** change the headline. `strict_v1` stays the
frozen rule of record — it produced every locked result and all of
`reference/*.json`. The looser rules are reported *alongside* it as a
sensitivity analysis. Moving a scoring rule after seeing that it raises
accuracy is the move this project keeps refusing to make.

In [ ]:
# Auth + code access. No GPU/model needed -- this notebook only reads the
# dataset and existing results CSVs, it never runs generation.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"

# Reuses the token already cached on Drive by earlier notebooks.
with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
# antlr4 pin: without it SymPy's LaTeX parser fails at CALL time, silently
# degrading every label to the plain-text tier. See
# pilot.canonicalize.latex_parser_available -- this cost 43/300 items once.
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))

# Purge any pilot.* left over from a previous clone in this runtime.
# importlib.invalidate_caches() does NOT reload already-imported modules, and
# a stale one produced a KeyError on the 2026-08-08 notebook-13 run for a
# symbol that was demonstrably on disk.
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.canonicalize
import pilot.data
import pilot.parsing
import pilot.plotting
import pilot.rescore

print(f"pilot package imported from: {os.path.dirname(pilot.rescore.__file__)}")
assert pilot.canonicalize.latex_parser_available(), (
    "SymPy's LaTeX parser is NOT working. Every label falls back to plain text, "
    "which inflates entropy and deflates accuracy, and the numbers below will "
    "not match the offline analysis. Fix the antlr4 pin before continuing."
)
print("SymPy LaTeX parser: OK")

## 1. Load both runs and rebuild the sample

Two models on the **same 300 images**: Qwen2.5-VL-3B (the run every scoring
number is quoted from) and Pixtral-12B (the confirmed second perception
family). Running the classification on both is what turns "the extractor has
problems" into "the extractor has problems that are not Qwen-specific".

The CSVs carry every raw model sample, so the whole scoring chain can be
re-run offline. The images are not in the CSV — they come from FERMAT, which
is gated, which is why this notebook runs in Colab rather than locally.

The `load_fermat_balanced` call reproduces the *same* draw both runs used, and
the assert below checks that row *i* of each CSV really is item *i* of the
sample rather than trusting the order.

In [ ]:
import ast

import pandas as pd

RUNS = {
    "Qwen2.5-VL-3B": "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv",
    "Pixtral-12B":   "pixtral_perception_full_n300_pixtral-12b_20260809T211028Z.csv",
}
SEED, N_ITEMS, ERROR_FRAC = 42, 300, 0.5

runs = {}
for name, fname in RUNS.items():
    runs[name] = pd.read_csv(f"{RESULTS_DIR}/{fname}")
    print(f"{name:15s} {len(runs[name])} rows, "
          f"model={runs[name]['model_id'].unique().tolist()}, "
          f"K={runs[name]['k_transcription'].unique().tolist()}")

sample = pilot.data.load_fermat_balanced(
    n=N_ITEMS, seed=SEED, target_error_frac=ERROR_FRAC)

# load_fermat_balanced SHUFFLES its final selection, so index alignment is an
# assumption to verify, not one to make. Checking the question text pins the
# row-to-image mapping every display below depends on -- for BOTH runs, since
# the viewer takes images from the sample and text from whichever CSV.
for name, df in runs.items():
    assert len(sample) == len(df), f"{name}: {len(sample)} items vs {len(df)} rows"
    bad = [i for i in range(len(df))
           if sample[i]["orig_q"].strip() != str(df.iloc[i]["orig_q"]).strip()]
    assert not bad, (
        f"{name}: {len(bad)} rows where the rebuilt sample's question does not "
        f"match the CSV's (first: {bad[:5]}). Images would be attached to the "
        "wrong rows -- do not trust anything past this cell until resolved.")
print("\nsample order matches both CSVs on all 300 rows -- images are index-aligned")

## 2. The sensitivity table

Four cumulative rules (see `pilot/rescore.py` for the full definitions):

- **`strict_v1`** — exactly `canonical_answer_label`. Frozen; the rule of record.
- **`fixed_v2`** — v1 with all three extractor bugs corrected. *Bug fixes*, so
  they move items **both ways** and can lower accuracy.
- **`relaxed_v3`** — v2 ignoring formatting the mathematics does not depend on.
- **`final_term_v4`** — v3 reduced to the term after the last top-level `=`.

v4 changes the **label**, not just the comparison, so entropy and correctness
stay derived from the same representation. Scoring correctness leniently while
leaving entropy strict would compare two different objects.

In [ ]:
# ~4 min: re-runs the full parse -> extract -> canonicalize chain for
# 2 models x 300 items x 5 samples x 4 rules, with SymPy on every label.
sens = {}
for name, df in runs.items():
    print(f"=== {name} ===")
    s = pilot.rescore.scoring_sensitivity(df, n_boot=10000, seed=0, progress=True)
    sens[name] = s
    view = s.copy()
    view["accuracy"] = (view["accuracy"] * 100).round(1).astype(str) + "%"
    view["AUROC [95% CI]"] = [f"{r.auroc:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]"
                              for r in s.itertuples()]
    print(view[["rule", "n_correct", "accuracy", "AUROC [95% CI]",
                "excludes_chance", "n_at_max_entropy"]].to_string(index=False))
    print(f"  accuracy {s.accuracy.iloc[0]:.1%} -> {s.accuracy.iloc[-1]:.1%}   "
          f"AUROC {s.auroc.iloc[0]:.3f} -> {s.auroc.iloc[-1]:.3f}   "
          f"every rule excludes chance: {bool(s.excludes_chance.all())}\n")

**Read it this way.** Accuracy moves a lot — the strict rule really is
undercounting correct reads. The AUROC decays gracefully and never touches
chance. A signal that existed only because of a pedantic string comparison
would not do that, so the perception result belongs to the entropy rather than
to the comparison.

## 3. Classify every item

The four illustrative buckets this notebook used to draw examples from
**overlapped** — an item could be both a cosmetic mismatch and
extractor-tier-unstable — so they could show you a case of X but could not say
what the population is made of. These seven categories are mutually exclusive
and cover every item.

| category | meaning |
|---|---|
| `correct_robust` | correct under the frozen rule **and** after the bug fixes |
| `bug_fix_recovered` | wrong under the frozen rule; a bug fix recovered it |
| `cosmetic_mismatch` | spacing, punctuation, braces, currency |
| `scope_mismatch` | model gave the answer, truth gave the whole chain |
| `false_pass_removed` | **looked correct only because a bug mangled both sides** |
| `broken_by_relaxation` | correct earlier, lost when a looser rule changed the vote |
| `genuinely_wrong` | wrong under every rule |

The last three exist because **the rules are not monotone**. A naive
cumulative scheme would hide them, and `false_pass_removed` is the one to read
first — it is where relaxing a comparison manufactures a wrong answer.

In [ ]:
classified, summaries = {}, []
for name, df in runs.items():
    c = pilot.rescore.classify_scoring_outcome(df, progress=True)
    classified[name] = c
    summaries.append(pilot.rescore.scoring_category_summary(c, label=name))
    assert len(c) == len(df) and c["category"].notna().all()

summary = pd.concat(summaries, ignore_index=True)

for name in runs:
    s = summary[summary.model == name]
    print(f"=== {name} ===")
    view = s[["category", "n", "share", "mean_entropy", "frac_multi_tier"]].copy()
    view["share"] = (view["share"] * 100).round(1).astype(str) + "%"
    view["mean_entropy"] = view["mean_entropy"].round(3)
    view["frac_multi_tier"] = (view["frac_multi_tier"] * 100).round(0).astype("Int64").astype(str) + "%"
    print(view.to_string(index=False))
    print(f"  total {int(s.n.sum())}   "
          f"later_regression (fixed then broken again): "
          f"{int(classified[name].later_regression.sum())}\n")

In [ ]:
# The bar plot: what the population is made of, both models side by side.
import matplotlib.pyplot as plt

FIG_DIR = f"{PROJECT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

ax = pilot.plotting.plot_scoring_categories(summary)
ax.set_title("What the 300 scored items are actually made of\n"
             "(blue = correct under the loosest rule, orange = not)",
             fontsize=10.5, loc="left")
ax.figure.tight_layout()
ax.figure.savefig(f"{FIG_DIR}/scoring_categories.png", dpi=160)
plt.show()
print("saved -> figures/scoring_categories.png")

### Which bug caused each flip?

`bug_fix_recovered` and `false_pass_removed` are attributed by applying each
fix **alone** and seeing which one changes the verdict. Small enough numbers to
name the responsible defect per item.

In [ ]:
for name, c in classified.items():
    attr = c["attributed_bug"].dropna()
    print(f"{name:15s} {attr.value_counts().to_dict()}")

print()
print("Cross-tab: attribution by category (Qwen)")
c = classified["Qwen2.5-VL-3B"]
print(pd.crosstab(c["category"], c["attributed_bug"].fillna("—")).to_string())

### The orthogonal flags

Extractor-tier instability is **not** a category — it cross-cuts all of them.
`extract_final_answer` has 4 tiers, and on a large share of items it fires a
*different* tier across the 5 samples, which inflates entropy without the model
having changed its mind.

**That count is a lower bound.** It only catches a changed *branch*. It misses
same-branch, different-block cases — item 9 has all five samples in
`display_math`, three returning the conclusion `(x,z) \in R` and two returning
the intermediate step, entropy 1.332 from samples that agree mathematically.

In [ ]:
for name, c in classified.items():
    print(f"=== {name} ===")
    print(f"  items using >1 extractor branch: "
          f"{int((c.n_distinct_tiers > 1).sum())}/{len(c)}")
    grp = c.groupby("n_distinct_tiers").agg(
        items=("entropy", "size"),
        mean_entropy=("entropy", "mean"),
        frac_correct_strict=("correct_strict_v1", "mean")).round(3)
    print(grp.to_string())
    ends_right = c.category.isin(["correct_robust", "bug_fix_recovered",
                                  "cosmetic_mismatch", "scope_mismatch"])
    print(f"  multi-tier among items that END correct: "
          f"{(c.n_distinct_tiers > 1)[ends_right].mean():.0%}")
    print(f"  multi-tier among items that END wrong  : "
          f"{(c.n_distinct_tiers > 1)[~ends_right].mean():.0%}")
    print("  -> cross-cuts the taxonomy, so it is a flag and not a bar\n")

## 4. The viewer

`show_item` prints the handwritten image and then the full four-stage trace,
rendered by `pilot.rescore.format_trace` so the naming lives beside the scoring
and cannot drift from it. Every line says which function produced it.

The **COMPARISON** block at the bottom is the part that was missing: on a
cosmetic mismatch the two labels look identical on screen, so it reports the
exact column where they first diverge and both characters by `repr`.

In [ ]:
IMAGE_DIR = f"{PROJECT_DIR}/scoring_inspection_images"
os.makedirs(IMAGE_DIR, exist_ok=True)


def show_item(i, model="Qwen2.5-VL-3B", rules=("strict_v1", "final_term_v4"),
              show_image=True, save=True, raw_chars=320):
    df = runs[model]
    row = df.iloc[i]
    cat = classified[model].loc[i]
    samples_raw = ast.literal_eval(row["all_transcription_samples_raw"])

    print("#" * 100)
    print(f"# ITEM {i}   [{model}]   category = {cat['category']}"
          + (f"   attributed to: {cat['attributed_bug']}"
             if pd.notna(cat["attributed_bug"]) else ""))
    print(f"#   has_error={bool(row['has_error'])}   "
          f"handwriting_style={row['handwriting_style']}   "
          f"image_quality={row['image_quality']}   "
          f"extractor branches used: {cat['n_distinct_tiers']}"
          + ("   later_regression=True" if cat["later_regression"] else ""))
    print("#   verdict by rule: " + "   ".join(
        f"{r}={bool(classified[model].loc[i, 'correct_' + r])}"
        for r in pilot.rescore.RULES))
    print("#" * 100)

    if show_image:
        img = sample[i]["image"]
        if save:
            img.save(f"{IMAGE_DIR}/item{i:03d}.png")
        plt.figure(figsize=(9, 9 * img.height / max(img.width, 1)))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"item {i} — the handwritten page the model was shown")
        plt.show()

    for rule in rules:
        tr = pilot.rescore.trace_item(samples_raw, row["pert_a"], rule)
        print(pilot.rescore.format_trace(
            tr, question=row["orig_q"], category=cat["category"],
            raw_chars=raw_chars))


def first_in(category, model="Qwen2.5-VL-3B", n=1):
    return classified[model].index[
        classified[model]["category"] == category].tolist()[:n]


print("show_item(i, model=...) ready.")
for name in runs:
    print(f"  {name}: " + "  ".join(
        f"{c}={len(classified[name][classified[name].category == c])}"
        for c in pilot.rescore.CATEGORIES))

### 4a. `false_pass_removed` — it only *looked* correct

Read this one first. A bug mangled the ground truth and the model's samples the
same way, so they matched — and the frozen rule scored it correct. It is the
proof that relaxing a comparison is not automatically generous.

In [ ]:
for i in first_in("false_pass_removed"):
    show_item(i)

### 4b. `cosmetic_mismatch` — the model read the page right

Watch the COMPARISON block: the two labels look identical until it names the
column where they diverge.

In [ ]:
for i in first_in("cosmetic_mismatch"):
    show_item(i)

### 4c. `scope_mismatch` — answer only vs the whole chain

In [ ]:
for i in first_in("scope_mismatch"):
    show_item(i)

### 4d. `bug_fix_recovered` — a real read the extractor was hiding

In [ ]:
for i in first_in("bug_fix_recovered"):
    show_item(i)

### 4e. `genuinely_wrong` — what a real misread looks like

The control for everything above. If these look like the categories above, the
taxonomy is not separating what it claims to.

In [ ]:
for i in first_in("genuinely_wrong"):
    show_item(i)

### 4f. The same categories on Pixtral-12B

Different model family, same failure modes — which is the point of running
both.

In [ ]:
for cat in ("false_pass_removed", "cosmetic_mismatch"):
    for i in first_in(cat, model="Pixtral-12B"):
        show_item(i, model="Pixtral-12B", rules=("strict_v1",))

## 5. Every item whose verdict a looser rule changes

For scanning after the detailed reads above. Shows the labels under the rule
that **failed** the item and the rule that **passed** it, so a "cosmetic" row
visibly has two differing labels on the left and one agreeing label on the right.

In [ ]:
FAIL_PASS = {"bug_fix_recovered": ("strict_v1", "fixed_v2"),
             "cosmetic_mismatch": ("fixed_v2", "relaxed_v3"),
             "scope_mismatch": ("relaxed_v3", "final_term_v4"),
             "false_pass_removed": ("fixed_v2", "strict_v1")}

MODEL = "Qwen2.5-VL-3B"
scored_by_rule = {r: pilot.rescore.rescore_run(runs[MODEL], r)
                  for r in pilot.rescore.RULES}
pd.set_option("display.max_colwidth", 46)

for cat, (fail_rule, pass_rule) in FAIL_PASS.items():
    idx = classified[MODEL].index[classified[MODEL].category == cat]
    if not len(idx):
        continue
    rows = [{
        "i": i,
        "H": round(float(classified[MODEL].loc[i, "entropy"]), 3),
        f"{fail_rule}: model": scored_by_rule[fail_rule].loc[i, "majority_label"][:44],
        f"{fail_rule}: truth": scored_by_rule[fail_rule].loc[i, "gt_label"][:44],
        f"{pass_rule}: both": scored_by_rule[pass_rule].loc[i, "majority_label"][:44],
    } for i in idx]
    print(f"--- {cat} ({len(rows)}) — differs under {fail_rule}, "
          f"agrees under {pass_rule} " + "-" * 18)
    print(pd.DataFrame(rows).to_string(index=False))
    print()

## 6. What to carry out of this notebook

- **`strict_v1` remains the reported rule.** Every locked number and every
  `reference/*.json` snapshot uses it, and it is bit-identical to
  `canonical_answer_label` (locked by
  `pilot/tests/test_rescore.py::test_strict_v1_is_bit_identical_to_the_frozen_pipeline`).
- **The transcription accuracy we report is a floor, not an estimate.** Say so
  in the paper, and quote it as a **range, 47–63%**, not as 63.3% — 9 of the 30
  items `final_term_v4` newly scores correct rest on a ≤2-character match
  (`1`, `5`, `24`), where a wrong answer can land on the same token by
  coincidence. The other 21 are substantive.
- **The AUROC is robust to the scoring rule**, which is the claim the
  sensitivity table actually supports.
- **The categories replicate across model families.** Qwen and Pixtral have the
  same shape — comparable `scope_mismatch` and an identical 6-item
  `false_pass_removed` — so these are properties of the *scoring pipeline*,
  not of one model.
- **All three extractor defects are real and are now implemented correctly** —
  `canonicalize.unwrap_latex_macro`, `extract_final_answer(...,
  fix_decimal_split=True)`, and `canonicalize_math(..., strict_parse=True)`.
  The frozen entry points keep the old behaviour on purpose, each with a
  docstring saying why.
- **Watch for `sympy:` labels containing spelled-out words** (`sympy:h*(e*(n*(c*e)))`).
  That is always SymPy having parsed English as multiplied variables and
  discarded the rest of the line.
- **Relaxation is not automatically generous.** `false_pass_removed` is 6 items
  in *both* models — items a looser rule scored correct only because a bug had
  mangled both sides the same way. Fix the extractor before trusting any
  relaxed number.
- **Open, and worth a sentence in Limitations:** the extractor fires different
  branches across the five samples on a large share of items, and mean entropy
  rises with that count. Part of the perception signal may be extractor
  instability rather than model uncertainty. Distinguishing them needs a rule
  where the extractor cannot vary — e.g. requiring `\boxed{}` in the prompt —
  which is a new run, not a rescoring.